<a href="https://colab.research.google.com/github/priyasrinivas781-code/The-syntax-surgeon/blob/main/Thesyntaxsurgeon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q gradio transformers torch reportlab


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 30.8 MB/s eta 0:00:00


In [ ]:
# Import libraries
import gradio as gr
from transformers import pipeline
import torch
from datetime import datetime
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

# Model configuration
MODEL_NAME = "ibm-granite/granite-3.3-2b-instruct"

# Detect device
device = 0 if torch.cuda.is_available() else -1

# Load model
print("Loading IBM Granite Model...")
model = pipeline(
    "text-generation",
    model=MODEL_NAME,
    device=device
)
print("Model Loaded!")

# Chat history storage
chat_history = []

# Chat function
def chat(user_message, temperature, max_tokens, top_p):

    response = model(
        user_message,
        max_new_tokens=int(max_tokens),
        temperature=temperature,
        top_p=top_p,
        do_sample=True
    )

    answer = response[0]["generated_text"]

    chat_history.append(("User", user_message))
    chat_history.append(("AI", answer))

    history_text = ""
    for role, msg in chat_history:
        history_text += f"{role}: {msg}\n\n"

    return history_text

# Download TXT
def download_txt():

    filename = "chat_history.txt"

    with open(filename, "w") as f:
        for role, msg in chat_history:
            f.write(f"{role}: {msg}\n\n")

    return filename

# Download PDF
def download_pdf():

    filename = "chat_history.pdf"

    styles = getSampleStyleSheet()
    elements = []

    for role, msg in chat_history:
        elements.append(Paragraph(f"<b>{role}</b>: {msg}", styles["Normal"]))
        elements.append(Spacer(1,10))

    doc = SimpleDocTemplate(filename)
    doc.build(elements)

    return filename

# Clear chat
def clear_chat():
    global chat_history
    chat_history = []
    return ""

# UI using Gradio
with gr.Blocks(title="Granite Chatbot") as demo:

    gr.Markdown("# Granite Chatbot")
    gr.Markdown("Chat + Python Code Fixer - No API Keys")

    with gr.Row():

        # Chat section
        with gr.Column(scale=3):

            chat_box = gr.Textbox(
                label="Chat History",
                lines=20
            )

            user_input = gr.Textbox(
                label="Message"
            )

            send_btn = gr.Button("Send")

        # Control panel
        with gr.Column(scale=1):

            gr.Markdown("### Controls")

            clear_btn = gr.Button("Clear Chat")

            gr.Markdown("### Download")

            txt_btn = gr.Button("Download TXT")
            pdf_btn = gr.Button("Download PDF")

            txt_file = gr.File(label="TXT File")
            pdf_file = gr.File(label="PDF File")

    with gr.Row():

        temperature = gr.Slider(0,1,value=0.7,label="Temp")
        max_tokens = gr.Slider(50,512,value=256,label="Tokens")
        top_p = gr.Slider(0,1,value=0.95,label="Top-P")

    send_btn.click(
        chat,
        inputs=[user_input, temperature, max_tokens, top_p],
        outputs=chat_box
    )

    txt_btn.click(download_txt, outputs=txt_file)

    pdf_btn.click(download_pdf, outputs=pdf_file)

    clear_btn.click(clear_chat, outputs=chat_box)

demo.launch()

Loading IBM Granite Model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/787 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/207 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model Loaded!
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://01391edf398e35478f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
